# Leçon 04 - Modèle de conception d'utilisation d'outil

Dans cette leçon, vous apprendrez le modèle de conception **Utilisation d'outil** pour les agents IA utilisant le Microsoft Agent Framework (Python). Nous couvrons :

- La définition d'outils fonctionnels avec le décorateur `@tool` et des paramètres typés
- La fourniture de schémas d'outil afin que le modèle sache ce que fait chaque outil
- Le contrôle de l'exécution des outils avec `approval_mode`
- Le retour de **sorties structurées** via des modèles Pydantic et `response_format`

Le scénario est un **agent de réservation de voyage** capable de rechercher des destinations, vérifier leur disponibilité et récupérer des informations sur les vols.


## Installation


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Définir des outils avec le décorateur @tool

Le décorateur `@tool` transforme une fonction Python simple en un outil qu'un agent peut appeler.  
Points clés :

- La **docstring** devient la description de l'outil que le modèle voit.  
- Les **annotations de type** (y compris `Annotated` avec des descriptions) définissent le schéma de l'outil.  
- `approval_mode` contrôle si l'utilisateur doit approuver chaque appel avant son exécution.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get available vacation destinations."""
    return ["Barcelona", "Paris", "Berlin", "Tokyo", "Sydney", "New York City"]


@tool(approval_mode="never_require")
def check_availability(
    destination: Annotated[str, "The destination to check"],
) -> str:
    """Check booking availability for a destination."""
    availability = {
        "Barcelona": "Available - 3 spots left",
        "Paris": "Available",
        "Berlin": "Sold out",
        "Tokyo": "Available - 1 spot left",
        "Sydney": "Available",
        "New York City": "Available",
    }
    return availability.get(destination, "Unknown destination")


@tool(approval_mode="never_require")
def get_flight_info(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
) -> str:
    """Get flight information between two cities."""
    flights = {
        "LHR-BCN": "BA 2042, Departs 08:30, Arrives 11:45, $350",
        "LHR-CDG": "AF 1081, Departs 09:15, Arrives 11:30, $280",
        "LHR-NRT": "JL 044, Departs 11:00, Arrives 07:00+1, $890",
    }
    return flights.get(
        f"{origin}-{destination}",
        f"No direct flights from {origin} to {destination}",
    )

## Création d'un agent avec plusieurs outils

Passez les trois outils au client afin que le modèle puisse invoquer ceux dont il a besoin pour répondre à la question de l'utilisateur.


In [ ]:
travel_tools = [get_destinations, check_availability, get_flight_info]

agent = await provider.create_agent(
    name="TravelToolAgent",
    instructions="You are a travel agent. Use the available tools to answer questions about destinations, availability, and flights.",
    tools=travel_tools,
)

response = await agent.run(
    "What destinations do you have? Which ones are still available?"
)
print(response)

## Sortie Structurée avec des Outils

En définissant `response_format` sur un modèle Pydantic, l'agent est obligé de retourner un objet JSON bien typé au lieu d'un texte libre. Ceci est utile lorsque le code en aval doit consommer le résultat de manière programmatique.


In [ ]:
class BookingRecommendation(BaseModel):
    destination: str
    available: bool
    flight_details: str
    estimated_cost: int


class TravelPlan(BaseModel):
    recommendations: list[BookingRecommendation]


structured_agent = await provider.create_agent(
    name="StructuredTravelAgent",
    instructions=(
        "You are a travel agent. Use the available tools to find destinations, "
        "check availability, and get flight info. Return structured results."
    ),
    tools=[get_destinations, check_availability, get_flight_info],
)

response = await structured_agent.run(
    "I want to fly from London Heathrow to somewhere warm in Europe. "
    "Check what's available."
)
if response:
    print(response)

## Modèles d'approbation des outils

Le paramètre `approval_mode` sur `@tool` contrôle si les appels d'outils nécessitent une approbation humaine avant l'exécution :

| Mode | Comportement |
|---|---|
| `"never_require"` | L'outil s'exécute automatiquement — aucune confirmation utilisateur n'est nécessaire. |
| `"always_require"` | Chaque appel doit être approuvé par l'utilisateur avant son exécution. |

Utilisez `"always_require"` pour les outils qui ont des effets secondaires (par exemple réserver un vol, facturer une carte de crédit) afin qu'un humain reste impliqué.


In [ ]:
@tool(approval_mode="always_require")
def book_flight(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
    passenger_name: Annotated[str, "Full name of the passenger"],
) -> str:
    """Book a flight for a passenger. Requires approval before executing."""
    return (
        f"Flight booked from {origin} to {destination} "
        f"for {passenger_name}. Confirmation #TRV-2024-{hash(passenger_name) % 10000:04d}"
    )


print("Tool name:", book_flight.name)
print("Approval mode:", book_flight.approval_mode)

## Résumé

Dans cette leçon, vous avez appris à :

1. **Définir des outils** en utilisant le décorateur `@tool` avec des paramètres typés et des docstrings qui servent de schéma pour l'outil.
2. **Composer plusieurs outils** afin que l'agent puisse les appeler en séquence pour répondre à des requêtes complexes.
3. **Retourner une sortie structurée** en passant un modèle Pydantic comme `response_format`.
4. **Contrôler l'approbation des outils** avec `approval_mode` pour maintenir un humain dans la boucle pour les opérations sensibles.

Ces modèles forment la base pour construire des agents fiables, prêts pour la production, capables d'interagir en toute sécurité avec des systèmes externes.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Avertissement** :  
Ce document a été traduit à l’aide du service de traduction automatique [Co-op Translator](https://github.com/Azure/co-op-translator). Bien que nous fassions tout notre possible pour assurer la précision, veuillez noter que les traductions automatiques peuvent contenir des erreurs ou des inexactitudes. Le document original dans sa langue d’origine doit être considéré comme la source faisant foi. Pour les informations critiques, il est recommandé de recourir à une traduction professionnelle réalisée par un humain. Nous déclinons toute responsabilité en cas de malentendus ou de mauvaises interprétations résultant de l’utilisation de cette traduction.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
